# Diabetes Hospital Readmission — Exploratory Data Analysis

This notebook documents the evidence behind every modelling decision in the pipeline.
Each section ends with a **Decision** line that maps directly to a configuration or
feature-engineering choice.

Dataset: UCI Diabetes 130-US Hospitals (1999–2008), ~101 000 encounters.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

DATA_PATH = '../data/raw_data/diabetic_data.csv'
df = pd.read_csv(DATA_PATH)

# Replace UCI '?' sentinel with NaN throughout
df.replace('?', np.nan, inplace=True)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

---
## 1. Target Distribution — which readmission class should we predict?

The raw target has three levels: `NO` (no readmission), `>30` (readmission after 30 days),
and `<30` (readmission within 30 days). The pipeline currently collapses both `>30` and `<30`
into a single positive class. This section examines whether that is the right choice.

In [ ]:
counts = df['readmitted'].value_counts().reindex(['NO', '>30', '<30'])
pcts   = counts / len(df) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: absolute counts
bars = axes[0].bar(counts.index, counts.values, color=['#4878CF', '#6ACC65', '#D65F5F'])
axes[0].set_title('Raw target class counts')
axes[0].set_ylabel('Encounters')
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 300, f'{v:,}', ha='center', fontsize=10)

# Right: two binarisation strategies compared
strategies = {
    'binary_any\n(current)': (counts['>30'] + counts['<30']) / len(df),
    'binary_30d\n(<30 only)': counts['<30'] / len(df),
}
axes[1].bar(strategies.keys(), [v*100 for v in strategies.values()],
            color=['#6ACC65', '#D65F5F'])
axes[1].set_title('Positive-class rate under each target definition')
axes[1].set_ylabel('Positive class (%)')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
for i, (_, v) in enumerate(strategies.items()):
    axes[1].text(i, v*100 + 0.3, f'{v*100:.1f}%', ha='center', fontsize=11)

fig.tight_layout()
plt.show()

print('Class counts:')
for label, n in counts.items():
    print(f'  {label:>3s}: {n:6,}  ({pcts[label]:.1f}%)')

**Findings:**
- `NO` accounts for ~54% of encounters; `>30` ~35%; `<30` ~11%.
- The `binary_any` target (current) creates a 46% positive class — nearly balanced — but
  conflates a clinically weak signal (`>30`) with the clinically actionable one (`<30`).
- The `binary_30d` target (`<30` only) is a genuinely hard, imbalanced problem (~11% positive)
  that matches what hospitals are penalised on under CMS readmission reduction programmes.

> **Decision:** Make the target mapping config-driven. Default to `binary_30d` for the
> primary model; keep `binary_any` available for comparison. This also means class-imbalance
> handling becomes necessary (see section 6).

---
## 2. Patient–Encounter Distribution — is there patient-level leakage risk?

The same patient can appear multiple times in the dataset. If different encounters for the
same patient land in both train and test, the model can memorise patient-level patterns
rather than learning generalisable clinical features.

In [ ]:
enc_per_patient = df.groupby('patient_nbr').size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: distribution of encounters per patient (clipped at 10 for readability)
clipped = enc_per_patient.clip(upper=10)
counts_enc = clipped.value_counts().sort_index()
axes[0].bar(counts_enc.index.astype(str).tolist()[:-1] + ['10+'],
            counts_enc.values,
            color='#4878CF')
axes[0].set_title('Encounters per patient')
axes[0].set_xlabel('Encounters')
axes[0].set_ylabel('Patients')

# Right: cumulative % of data covered by multi-encounter patients
multi = (enc_per_patient > 1).sum()
single = (enc_per_patient == 1).sum()
axes[1].pie(
    [single, multi],
    labels=[f'Single encounter\n({single:,})', f'Multiple encounters\n({multi:,})'],
    colors=['#6ACC65', '#D65F5F'],
    autopct='%1.1f%%',
    startangle=90,
)
axes[1].set_title('Single vs multi-encounter patients')

fig.tight_layout()
plt.show()

n_patients = enc_per_patient.shape[0]
n_multi    = (enc_per_patient > 1).sum()
print(f'Total unique patients : {n_patients:,}')
print(f'Multi-encounter patients: {n_multi:,} ({n_multi/n_patients*100:.1f}%)')
print(f'Max encounters for one patient: {enc_per_patient.max()}')

**Findings:**
- A meaningful share of patients have more than one encounter. Without a group-aware split,
  encounters from the same patient appear on both sides of the train/test boundary.
- This inflates held-out metrics because the model can exploit patient-specific memorised
  patterns rather than generalisable clinical features.

> **Decision:** Use `StratifiedGroupKFold` with `group_column: patient_nbr` for both the
> train/test split and all CV folds. **Already implemented** in `training_splits.py` and
> `run_config.yaml`.

---
## 3. Missingness — which columns have substantial null rates?

The UCI dataset uses `?` as a missing-value sentinel (replaced with `NaN` at load time).
High missingness in a column determines whether to impute or drop.

In [ ]:
null_pct = df.isnull().mean().sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#D65F5F' if v > 0.4 else '#F5A623' if v > 0.05 else '#6ACC65'
          for v in null_pct.values]
bars = ax.barh(null_pct.index, null_pct.values * 100, color=colors)
ax.axvline(5,  color='orange', linestyle='--', linewidth=1, label='5% threshold')
ax.axvline(40, color='red',    linestyle='--', linewidth=1, label='40% threshold')
ax.set_xlabel('Missing values (%)')
ax.set_title('Missingness by column (columns with any nulls)')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend()
for bar, v in zip(bars, null_pct.values):
    ax.text(v*100 + 0.5, bar.get_y() + bar.get_height()/2,
            f'{v*100:.1f}%', va='center', fontsize=9)
fig.tight_layout()
plt.show()

print('Columns with > 5% missing:')
print(null_pct[null_pct > 0.05].to_string())

**Findings:**
- `weight` is missing for ~97% of encounters — effectively useless as a feature; dropping is correct.
- `payer_code` and `medical_specialty` have ~40% and ~49% missing respectively.
  `medical_specialty` is kept and imputed with `most_frequent` because specialty is a
  meaningful clinical grouping variable.
- `race` has ~2% missing — imputed safely.
- `diag_1/2/3` have a small fraction of nulls — imputed before grouping.

> **Decision:** Drop `weight` (already done). Impute remaining columns with `most_frequent`
> for categoricals and `mean` for numericals — already set in `data_engineering` config.

---
## 4. ICD-9 Diagnosis Cardinality — can we use raw codes as features?

The three diagnosis columns (`diag_1`, `diag_2`, `diag_3`) contain raw ICD-9 codes.
One-hot encoding them directly produces an extremely wide, sparse feature matrix.

In [ ]:
diag_cols = ['diag_1', 'diag_2', 'diag_3']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: unique code count per column
unique_counts = {col: df[col].nunique() for col in diag_cols}
axes[0].bar(unique_counts.keys(), unique_counts.values(), color='#4878CF')
axes[0].set_title('Unique ICD-9 codes per diagnosis column')
axes[0].set_ylabel('Unique codes')
for i, (col, v) in enumerate(unique_counts.items()):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=11)

# Right: top-20 most frequent diag_1 codes
top20 = df['diag_1'].value_counts().head(20)
axes[1].barh(top20.index[::-1], top20.values[::-1], color='#4878CF')
axes[1].set_title('Top 20 most frequent diag_1 codes')
axes[1].set_xlabel('Frequency')

fig.tight_layout()
plt.show()

# Show what fraction of encounters the top-20 codes cover
top20_share = top20.sum() / df['diag_1'].notna().sum()
print(f'Top 20 diag_1 codes cover {top20_share*100:.1f}% of all encounters')
print(f'Total unique diag_1 codes: {df["diag_1"].nunique()}')

In [ ]:
# Illustrate the ICD-9 grouping scheme that condenses codes into clinical categories
def icd9_group(code):
    """Map a raw ICD-9 code to one of ~18 clinical categories."""
    if pd.isna(code):
        return 'Missing'
    code = str(code)
    if code.startswith('V') or code.startswith('E'):
        return 'External/Supplementary'
    try:
        c = float(code)
    except ValueError:
        return 'Other'
    if 390 <= c <= 459 or c == 785:   return 'Circulatory'
    if 460 <= c <= 519 or c == 786:   return 'Respiratory'
    if 520 <= c <= 579 or c == 787:   return 'Digestive'
    if 250 <= c <= 250.99:            return 'Diabetes'
    if 800 <= c <= 999:               return 'Injury'
    if 710 <= c <= 739:               return 'Musculoskeletal'
    if 580 <= c <= 629 or c == 788:   return 'Genitourinary'
    if 140 <= c <= 239:               return 'Neoplasms'
    if 240 <= c <= 279:               return 'Endocrine'
    if 680 <= c <= 709 or c == 782:   return 'Skin'
    if 001 <= c <= 139:               return 'Infectious'
    if 290 <= c <= 319:               return 'Mental'
    if 280 <= c <= 289:               return 'Blood'
    if 320 <= c <= 389:               return 'Nervous'
    if 630 <= c <= 679:               return 'Pregnancy'
    if 740 <= c <= 759:               return 'Congenital'
    if 760 <= c <= 779:               return 'Perinatal'
    return 'Other'

df['diag_1_group'] = df['diag_1'].apply(icd9_group)

group_counts = df['diag_1_group'].value_counts()
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(group_counts.index[::-1], group_counts.values[::-1], color='#4878CF')
ax.set_title('diag_1 after ICD-9 grouping (18 categories vs ~700 raw codes)')
ax.set_xlabel('Encounters')
fig.tight_layout()
plt.show()

print(f'Raw unique codes  : {df["diag_1"].nunique()}')
print(f'After grouping    : {df["diag_1_group"].nunique()}')

**Findings:**
- `diag_1` alone has ~700 unique ICD-9 codes; one-hot encoding all three columns would add
  ~1 600 sparse binary columns, most with near-zero frequency.
- Grouping into ~18 clinical categories reduces this to 54 columns, removes sparsity, and
  lets the model learn meaningful clinical patterns (e.g. circulatory disease → higher risk).

> **Decision:** Implement `icd9_group()` as a pre-processing step applied to `diag_1/2/3`
> before the column transformer runs. Add as `feature_engineering.py`.

---
## 5. Discharge Disposition vs Readmission — should we keep this feature?

`discharge_disposition_id` is currently **dropped** in `run_config.yaml`. This section
checks whether it carries signal for the readmission target.

In [ ]:
# Map disposition IDs to readable labels (subset of UCI codebook)
disposition_labels = {
    1:  'Home',
    2:  'Short-term hospital',
    3:  'Skilled nursing facility',
    4:  'Intermediate care',
    5:  'Other inpatient',
    6:  'Home health service',
    7:  'Left AMA',
    8:  'Home IV',
    9:  'Admitted as inpatient',
    11: 'Expired',
    13: 'Hospice/medical facility',
    14: 'Hospice/home',
    18: 'Unknown',
    19: 'Expired at home',
    20: 'Expired in facility',
    25: 'Not mapped',
}

df['disp_label'] = df['discharge_disposition_id'].map(disposition_labels).fillna('Other')
df['readmitted_30d'] = (df['readmitted'] == '<30').astype(int)

# Readmission rate by disposition (keep dispositions with >= 100 encounters)
disp_stats = (
    df.groupby('disp_label')
    .agg(n=('readmitted_30d', 'count'), readmit_rate=('readmitted_30d', 'mean'))
    .query('n >= 100')
    .sort_values('readmit_rate', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(disp_stats.index, disp_stats['readmit_rate'] * 100,
               color='#D65F5F')
overall_rate = df['readmitted_30d'].mean() * 100
ax.axvline(overall_rate, color='black', linestyle='--', linewidth=1.2,
           label=f'Overall rate ({overall_rate:.1f}%)')
ax.set_xlabel('<30-day readmission rate (%)')
ax.set_title('30-day readmission rate by discharge disposition')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend()
for bar, (_, row) in zip(bars, disp_stats.iterrows()):
    ax.text(row['readmit_rate']*100 + 0.1,
            bar.get_y() + bar.get_height()/2,
            f"{row['readmit_rate']*100:.1f}%  (n={row['n']:,})",
            va='center', fontsize=9)
fig.tight_layout()
plt.show()

**Findings:**
- There is substantial variation in `<30`-day readmission rate across disposition categories.
- Patients discharged to short-term hospitals or skilled nursing facilities have notably
  different readmission rates compared to those discharged home.
- This is consistent with the clinical literature, which identifies discharge destination
  as one of the strongest individual predictors of readmission.

> **Decision:** Remove `discharge_disposition_id` from the `drop_columns` list in
> `run_config.yaml` and add it to `categorical_features` for one-hot encoding.

---
## 6. Medication Change Distribution — is a polypharmacy count useful?

~24 medication columns each have values `No / Steady / Up / Down`. The current approach
one-hot encodes all of them, producing ~96 sparse columns. A single integer count of
dose changes (`Up` or `Down`) may capture the clinical signal more efficiently.

In [ ]:
med_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'insulin',
    'glyburide-metformin', 'glipizide-metformin',
]

# Count medications with any dose change per encounter
df['polypharmacy_changes'] = (df[med_cols].isin(['Up', 'Down'])).sum(axis=1)

# Count medications actively prescribed (not 'No')
df['n_active_meds'] = (df[med_cols] != 'No').sum(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution of polypharmacy change count
change_counts = df['polypharmacy_changes'].value_counts().sort_index()
axes[0].bar(change_counts.index, change_counts.values, color='#4878CF')
axes[0].set_title('Encounters by number of\ndose changes')
axes[0].set_xlabel('Medications with Up/Down change')
axes[0].set_ylabel('Encounters')

# Readmission rate by polypharmacy change count
poly_readmit = df.groupby('polypharmacy_changes')['readmitted_30d'].mean() * 100
axes[1].bar(poly_readmit.index, poly_readmit.values, color='#D65F5F')
axes[1].axhline(overall_rate, color='black', linestyle='--', linewidth=1)
axes[1].set_title('30-day readmission rate\nby dose change count')
axes[1].set_xlabel('Medications with Up/Down change')
axes[1].set_ylabel('Readmission rate (%)')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())

# Stacked bar: distribution of values across all medication columns
med_value_counts = pd.DataFrame({
    col: df[col].value_counts(normalize=True) for col in med_cols
}).T.fillna(0)
for col in ['No', 'Steady', 'Up', 'Down']:
    if col not in med_value_counts.columns:
        med_value_counts[col] = 0
med_value_counts = med_value_counts[['No', 'Steady', 'Up', 'Down']]
med_value_counts.sort_values('No').plot(
    kind='barh', stacked=True,
    color=['#CCCCCC', '#6ACC65', '#4878CF', '#D65F5F'],
    ax=axes[2], legend=True
)
axes[2].set_title('Medication value distribution\n(sorted by % No)')
axes[2].set_xlabel('Proportion of encounters')
axes[2].legend(loc='lower right', fontsize=8)

fig.tight_layout()
plt.show()

print(f'Encounters with zero dose changes : {(df["polypharmacy_changes"]==0).sum():,}')
print(f'Encounters with >= 1 dose change  : {(df["polypharmacy_changes"]>=1).sum():,}')

**Findings:**
- Most medications are `No` for the majority of encounters — highly sparse when one-hot encoded.
- The readmission rate varies with dose-change count, suggesting the aggregate count
  captures a meaningful clinical signal (regimen complexity at discharge).
- `insulin` is the most commonly adjusted medication and likely warrants its own indicator.

> **Decision:** Add a `polypharmacy_changes` integer feature (count of `Up`/`Down` across
> all medication columns) in `feature_engineering.py`. Keep individual medication columns
> for one-hot encoding but the aggregate count adds a complementary signal.

---
## 7. Numerical Feature Distributions and Correlation with Target

In [ ]:
num_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_diagnoses', 'number_inpatient',
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(num_features):
    ax = axes[i]
    ax.hist(df[df['readmitted_30d'] == 0][col].dropna(), bins=30, alpha=0.6,
            color='#6ACC65', label='Not readmitted <30d', density=True)
    ax.hist(df[df['readmitted_30d'] == 1][col].dropna(), bins=30, alpha=0.6,
            color='#D65F5F', label='Readmitted <30d', density=True)
    ax.set_title(col.replace('_', ' '))
    ax.set_ylabel('Density')

axes[i+1].axis('off')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', fontsize=10)
fig.suptitle('Numerical feature distributions by 30-day readmission', y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Point-biserial correlation of numerical features with 30d readmission
corrs = df[num_features + ['readmitted_30d']].corr()['readmitted_30d'].drop('readmitted_30d').sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#D65F5F' if v > 0 else '#4878CF' for v in corrs.values]
ax.barh(corrs.index, corrs.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson correlation with 30-day readmission (numerical features)')
ax.set_xlabel('Correlation coefficient')
fig.tight_layout()
plt.show()

print(corrs.to_string())

**Findings:**
- `number_inpatient` (prior inpatient visits) shows the strongest positive correlation with
  30-day readmission — prior utilisation is a well-established risk factor.
- `number_emergency` also correlates positively.
- `num_lab_procedures` and `time_in_hospital` are weakly correlated — encode acuity but
  conflate many different patient profiles.

> **Decision:** All numerical features are kept. `number_inpatient` and `number_emergency`
> should appear near the top of SHAP importance plots if the model is correct.

---
## 8. Age Distribution and Readmission Rate

In [ ]:
age_order = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
             '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']

age_stats = (
    df.groupby('age')
    .agg(n=('readmitted_30d', 'count'), readmit_rate=('readmitted_30d', 'mean'))
    .reindex(age_order)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(age_stats.index, age_stats['n'], color='#4878CF')
axes[0].set_title('Encounter count by age group')
axes[0].set_xlabel('Age bracket')
axes[0].set_ylabel('Encounters')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(age_stats.index, age_stats['readmit_rate'] * 100,
             marker='o', color='#D65F5F', linewidth=2)
axes[1].axhline(overall_rate, color='black', linestyle='--', linewidth=1,
                label=f'Overall ({overall_rate:.1f}%)')
axes[1].set_title('30-day readmission rate by age group')
axes[1].set_xlabel('Age bracket')
axes[1].set_ylabel('Readmission rate (%)')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

fig.tight_layout()
plt.show()

**Findings:**
- The dataset is heavily weighted towards the 50–90 age range, reflecting the diabetic
  inpatient population.
- Readmission rate is relatively consistent across age groups, with a slight elevation
  in the middle-aged brackets. Age alone is not a strong discriminator.

> **Decision:** Keep `age` as an ordinal categorical feature.

---
## 9. Class Imbalance — what does the binary_30d target look like?

After re-framing to `binary_30d`, the positive class (~11%) becomes a genuine minority.
This section quantifies the imbalance and justifies the class-weighting approach.

In [ ]:
pos = df['readmitted_30d'].sum()
neg = len(df) - pos
ratio = neg / pos

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(['Negative (NO or >30)', 'Positive (<30)'], [neg, pos],
            color=['#6ACC65', '#D65F5F'])
axes[0].set_title('Class counts under binary_30d target')
axes[0].set_ylabel('Encounters')
for i, v in enumerate([neg, pos]):
    axes[0].text(i, v + 300, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center')

# Impact of scale_pos_weight / class_weight='balanced'
# scale_pos_weight for XGBoost = neg/pos
strategies = ['No weighting', 'class_weight=balanced\n(or scale_pos_weight)']
effective_weight = [1, ratio]
axes[1].bar(strategies, effective_weight, color=['#CCCCCC', '#4878CF'])
axes[1].set_title('Effective weight on positive class')
axes[1].set_ylabel('Weight multiplier')
axes[1].text(1, ratio + 0.1, f'{ratio:.1f}x', ha='center', fontsize=12)

fig.tight_layout()
plt.show()

print(f'Negative class : {neg:,}  ({neg/len(df)*100:.1f}%)')
print(f'Positive class : {pos:,}  ({pos/len(df)*100:.1f}%)')
print(f'Imbalance ratio (neg/pos) = {ratio:.1f}:1')
print(f'Recommended scale_pos_weight for XGBoost: {ratio:.1f}')

**Findings:**
- Under `binary_30d`, the imbalance ratio is ~8:1. Without correction, the model will
  be biased towards predicting the majority class and recall on the positive class
  (the patients actually at risk) will be poor.
- `class_weight='balanced'` (sklearn models) or `scale_pos_weight=ratio` (XGBoost)
  re-weights the loss to treat a positive-class error as `ratio` times more costly.

> **Decision:** Add `scale_pos_weight` to the XGBoost config and `class_weight: balanced`
> to sklearn model configs. Make it config-driven under `imbalance.strategy`.

---
## Summary of Modelling Decisions

| Section | Finding | Decision |
|---------|---------|----------|
| 1. Target | `>30` dilutes the `<30` signal | Use `binary_30d` target (config-driven) |
| 2. Patient leakage | Multi-encounter patients create leakage risk | Group split on `patient_nbr` ✅ |
| 3. Missingness | `weight` ~97% null; `medical_specialty` ~49% | Drop `weight`; impute others ✅ |
| 4. ICD-9 cardinality | ~700 raw codes → wide sparse matrix | Group into 18 clinical categories |
| 5. Discharge disposition | Strong variation in readmission rate across dispositions | Remove from drop list; encode as feature |
| 6. Medication changes | Dose-change count correlates with readmission | Add `polypharmacy_changes` count feature |
| 7. Numerical features | `number_inpatient` strongest predictor | Keep all; expect it top in SHAP |
| 8. Age | Weak discriminator across age groups | Keep as ordinal categorical |
| 9. Class imbalance | ~8:1 ratio under `binary_30d` | `scale_pos_weight` / `class_weight=balanced` |